# Notebook 04: Exploratory Data Analysis & Visualization

**Mục tiêu:** Khám phá dữ liệu nhiệt độ bề mặt Trái Đất, phát hiện các xu hướng, mẫu mùa vụ, và mối liên hệ giữa các biến để xây dựng nền tảng cho Feature Engineering.

**Quy trình:** Notebook 03 (Data Cleaning) → **Notebook 04 (EDA & Visualization)** → Notebook 05 (Feature Engineering)

---

## I. Giới Thiệu

### Mục tiêu của EDA

Exploratory Data Analysis (EDA) giúp:
1. **Hiểu sâu dữ liệu:** Phân bố, xu hướng, mối quan hệ giữa các biến.
2. **Phát hiện các yếu tố ảnh hưởng:** Xác định các cột có tác động mạnh đến biến mục tiêu hoặc các xu hướng chính.
3. **Chuẩn bị cho Feature Engineering:** Những phát hiện này sẽ được sử dụng để tạo các đặc trưng mới.
4. **Nhận diện vấn đề tiềm ẩn:** Outliers, missing values không được xử lý, hoặc các bất thường cần lưu ý.

### 4 Câu hỏi cốt lõi mà Notebook 04 trả lời

1. **Dữ liệu có đặc điểm gì?** (phân bố, xu hướng, mối quan hệ)
2. **Những yếu tố nào ảnh hưởng đến nhiệt độ?** (thông qua trực quan hóa và phân tích)
3. **Kết quả có ý nghĩa gì đối với bài toán dự báo?** (không chỉ mô tả biểu đồ mà còn giải thích ý nghĩa)
4. **Những phát hiện này sẽ được sử dụng như thế nào ở bước tiếp theo?** (nền tảng cho Feature Engineering)

---

## II. Đọc Dữ Liệu

### 2.1 Import thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ Thư viện đã được import")

✓ Thư viện đã được import


### 2.2 Xác định đường dẫn dữ liệu

In [2]:
# Tìm project root dựa trên thư mục 'data' và 'notebooks'
def find_project_root():
    current = Path.cwd()
    while current != current.parent:
        if (current / 'data').exists() and (current / 'notebooks').exists():
            return current
        current = current.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
REPORT_IMAGES = PROJECT_ROOT / 'reports' / 'images'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Processed: {DATA_PROCESSED}")
print(f"Report Images: {REPORT_IMAGES}")

Project Root: c:\Users\COMPUTER\Desktop\du_an_1\Global-Surface-Temperature-Analysis
Data Processed: c:\Users\COMPUTER\Desktop\du_an_1\Global-Surface-Temperature-Analysis\data\processed
Report Images: c:\Users\COMPUTER\Desktop\du_an_1\Global-Surface-Temperature-Analysis\reports\images


### 2.3 Đọc dữ liệu từ PostgreSQL hoặc CSV

In [ ]:
from dotenv import load_dotenv
import os
import time

DATA_SOURCE = 'CSV'
df = None

# Tải .env ngay từ đầu (nếu tồn tại)
env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    load_dotenv(env_path)

DB_HOST = os.getenv('DB_HOST', '')
DB_PORT = os.getenv('DB_PORT', '')
DB_NAME = os.getenv('DB_NAME', '')
DB_USER = os.getenv('DB_USER', '')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')

# Chỉ thử PostgreSQL nếu có đầy đủ thông tin
if all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    try:
        import sqlalchemy
        from sqlalchemy.pool import NullPool
        
        print("🔌 Đang kết nối PostgreSQL (5s timeout)...")
        start_time = time.time()
        
        # Connection string với timeout ngắn
        connection_string = (
            f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
        )
        
        # Engine với timeout và pool settings tối ưu
        engine = sqlalchemy.create_engine(
            connection_string,
            connect_args={'connect_timeout': 5},  # Timeout 5 giây
            poolclass=NullPool,  # Không cache connection
            echo=False
        )
        
        # Kiểm tra kết nối nhanh
        with engine.connect() as conn:
            conn.execute(sqlalchemy.text('SELECT 1'))
        
        # Đọc dữ liệu
        df = pd.read_sql(
            'SELECT * FROM cleaned_city_temperature',
            engine
        )
        elapsed = time.time() - start_time
        DATA_SOURCE = 'POSTGRESQL'
        print(f"✓ Dữ liệu đã được đọc từ PostgreSQL ({elapsed:.2f}s)")
        engine.dispose()
        
    except TimeoutError:
        print("⏱️ PostgreSQL timeout (quá lâu để kết nối)")
        print("  Fallback sang CSV...")
    except Exception as e:
        error_type = type(e).__name__
        print(f"ℹ PostgreSQL không khả dụng ({error_type})")
        print("  Fallback sang CSV...")
else:
    if not all([DB_HOST, DB_PORT, DB_NAME, DB_USER]):
        print("ℹ Không có cấu hình PostgreSQL (.env không đầy đủ)")

# Fallback: đọc từ CSV
if df is None:
    csv_path = DATA_PROCESSED / 'cleaned_city_temperature.csv'
    if csv_path.exists():
        print("📂 Đang đọc từ CSV...")
        start_time = time.time()
        df = pd.read_csv(csv_path)
        elapsed = time.time() - start_time
        DATA_SOURCE = 'CSV (data/processed)'
        print(f"✓ Dữ liệu đã được đọc từ {csv_path} ({elapsed:.2f}s)")
    else:
        print(f"❌ Không tìm thấy {csv_path}")
        print(f"   Các file có sẵn trong {DATA_PROCESSED}:")
        if DATA_PROCESSED.exists():
            for f in list(DATA_PROCESSED.glob('*.csv'))[:10]:
                print(f"   - {f.name}")

print(f"\n📊 Nguồn dữ liệu: {DATA_SOURCE}")

🔌 Đang kết nối PostgreSQL (5s timeout)...


### 2.4 Kiểm tra dữ liệu cơ bản

In [ ]:
if df is not None:
    print(f"📌 Kích thước dữ liệu: {df.shape}")
    print(f"   - Số dòng: {df.shape[0]:,}")
    print(f"   - Số cột: {df.shape[1]}")
    
    print(f"\n📋 Tên các cột:")
    for i, col in enumerate(df.columns, 1):
        print(f"   {i}. {col}")
    
    print(f"\n📊 Kiểu dữ liệu:")
    print(df.dtypes)
    
    print(f"\n📈 Mẫu dữ liệu (5 dòng đầu):")
    display(df.head())
    
    print(f"\n📉 Thống kê cơ bản (số liệu):")
    display(df.describe())
else:
    print("❌ Không thể tải dữ liệu. Vui lòng kiểm tra lại đường dẫn hoặc PostgreSQL.")

📌 Kích thước dữ liệu: (5579085, 32)
   - Số dòng: 5,579,085
   - Số cột: 32

📋 Tên các cột:
   1. observation_date
   2. year
   3. month
   4. quarter
   5. decade
   6. city_name
   7. country_name
   8. latitude
   9. longitude
   10. is_major_city
   11. city_average_temperature
   12. city_average_temperature_uncertainty
   13. country_matched
   14. country_average_temperature
   15. country_average_temperature_uncertainty
   16. global_matched
   17. land_average_temperature
   18. land_average_temperature_uncertainty
   19. land_max_temperature
   20. land_max_temperature_uncertainty
   21. land_min_temperature
   22. land_min_temperature_uncertainty
   23. land_and_ocean_average_temperature
   24. land_and_ocean_average_temperature_uncertainty
   25. major_city_matched
   26. major_city_average_temperature
   27. major_city_average_temperature_uncertainty
   28. city_uncertainty_available
   29. country_temperature_available
   30. global_temperature_available
   31. major_cit

,observation_date,year,month,quarter,decade,city_name,country_name,latitude,longitude,is_major_city,city_average_temperature,city_average_temperature_uncertainty,country_matched,country_average_temperature,country_average_temperature_uncertainty,global_matched,land_average_temperature,land_average_temperature_uncertainty,land_max_temperature,land_max_temperature_uncertainty,land_min_temperature,land_min_temperature_uncertainty,land_and_ocean_average_temperature,land_and_ocean_average_temperature_uncertainty,major_city_matched,major_city_average_temperature,major_city_average_temperature_uncertainty,city_uncertainty_available,country_temperature_available,global_temperature_available,major_city_temperature_available,city_temperature_iqr_outlier
0,1863-01-01,1863,1,1,1860,A Coruña,Spain,42.59,-8.73,False,8.131,2.430,True,5.923,2.171,True,3.034,1.228,8.462,3.188,-1.766,2.51,13.296,0.417,False,NaN,NaN,True,True,True,False,False
1,1863-01-01,1863,1,1,1860,Aachen,Germany,50.63,6.34,False,2.992,2.230,True,2.049,2.282,True,3.034,1.228,8.462,3.188,-1.766,2.51,13.296,0.417,False,NaN,NaN,True,True,True,False,False
2,1863-01-01,1863,1,1,1860,Abadan,Iran,29.74,48.00,False,10.934,1.847,True,4.057,1.889,True,3.034,1.228,8.462,3.188,-1.766,2.51,13.296,0.417,False,NaN,NaN,True,True,True,False,False
3,1863-01-01,1863,1,1,1860,Abakan,Russia,53.84,91.36,False,-17.532,2.227,True,-24.312,1.862,True,3.034,1.228,8.462,3.188,-1.766,2.51,13.296,0.417,False,NaN,NaN,True,True,True,False,True
4,1863-01-01,1863,1,1,1860,Abbotsford,Canada,49.03,-122.45,False,-0.377,3.315,True,-22.554,2.829,True,3.034,1.228,8.462,3.188,-1.766,2.51,13.296,0.417,False,NaN,NaN,True,True,True,False,False



📉 Thống kê cơ bản (số liệu):


,year,month,quarter,decade,latitude,longitude,city_average_temperature,city_average_temperature_uncertainty,country_average_temperature,country_average_temperature_uncertainty,land_average_temperature,land_average_temperature_uncertainty,land_max_temperature,land_max_temperature_uncertainty,land_min_temperature,land_min_temperature_uncertainty,land_and_ocean_average_temperature,land_and_ocean_average_temperature_uncertainty,major_city_average_temperature,major_city_average_temperature_uncertainty
count,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.572950e+06,5.572950e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,5.579085e+06,151293.000000,151293.000000
mean,1.938584e+03,6.491461e+00,2.497298e+00,1.934093e+03,2.346982e+01,3.496685e+01,1.750208e+01,5.836120e-01,1.567140e+01,4.558198e-01,8.609823e+00,2.325599e-01,1.439795e+01,3.445473e-01,2.788128e+00,3.431562e-01,1.523108e+01,1.149765e-01,18.239882,0.589568
std,4.313090e+01,3.450317e+00,1.117522e+00,4.328725e+01,2.313193e+01,7.676315e+01,1.019355e+01,4.461194e-01,1.148021e+01,3.662704e-01,4.246959e+00,1.609864e-01,4.324784e+00,3.262957e-01,4.157543e+00,3.062717e-01,1.265668e+00,5.762411e-02,9.925407,0.448648
min,1.863000e+03,1.000000e+00,1.000000e+00,1.860000e+03,-5.224000e+01,-1.511300e+02,-4.270400e+01,3.400000e-02,-3.049700e+01,5.200000e-02,5.000000e-01,3.400000e-02,5.900000e+00,4.400000e-02,-5.345000e+00,4.500000e-02,1.262000e+01,4.200000e-02,-26.772000,0.040000
25%,1.901000e+03,3.000000e+00,1.000000e+00,1.900000e+03,1.045000e+01,-7.540000e+00,1.160600e+01,2.930000e-01,8.493000e+00,2.130000e-01,4.488000e+00,9.600000e-02,1.027900e+01,1.360000e-01,-1.308000e+00,1.480000e-01,1.406200e+01,6.200000e-02,12.945000,0.294000
50%,1.939000e+03,6.000000e+00,2.000000e+00,1.930000e+03,2.813000e+01,4.364000e+01,1.982600e+01,4.360000e-01,1.885500e+01,3.150000e-01,8.913000e+00,2.160000e-01,1.478600e+01,2.340000e-01,2.994000e+00,2.580000e-01,1.528700e+01,1.190000e-01,20.619000,0.439000
75%,1.976000e+03,9.000000e+00,3.000000e+00,1.970000e+03,3.938000e+01,1.062200e+02,2.563700e+01,7.200000e-01,2.510100e+01,5.790000e-01,1.286300e+01,3.070000e-01,1.871000e+01,4.110000e-01,6.822000e+00,3.790000e-01,1.640600e+01,1.410000e-01,25.950000,0.730000
max,2.013000e+03,1.200000e+01,4.000000e+00,2.010000e+03,6.992000e+01,1.595500e+02,3.915600e+01,1.269400e+01,3.314800e+01,4.348000e+00,1.548200e+01,1.228000e+00,2.132000e+01,3.188000e+00,9.715000e+00,2.510000e+00,1.760900e+01,4.170000e-01,36.477000,5.677000


---

## III. Khám Phá Dữ Liệu (Exploratory Data Analysis)

### 3.1 Correlation Analysis (Phân tích tương quan)

#### 3.1.1 Ma trận tương quan

In [ ]:
if df is not None:
    # Lọc các cột số
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        corr_matrix = df[numeric_cols].corr()
        
        print(f"📊 Ma trận tương quan giữa các biến số:")
        print(corr_matrix)
        
        # Heatmap - kích thước phụ thuộc vào số lượng biến
        n_vars = len(corr_matrix)
        fig_size = max(8, min(16, n_vars * 0.8 + 2))
        plt.figure(figsize=(fig_size, fig_size * 0.95))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                    center=0, square=True, cbar_kws={'label': 'Correlation'})
        plt.title('Correlation Matrix - Biểu đồ Tương Quan', fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '01_correlation_heatmap.png', dpi=300, bbox_inches='tight')
            print("✓ Đã lưu: reports/images/01_correlation_heatmap.png")
        
        plt.show()
    else:
        print("⚠ Không có cột số để tính tương quan.")

#### 3.1.2 Nhận xét tương quan

**Quan sát chính:**
- Các biến nhiệt độ thường có tương quan cao với nhau (ví dụ: land_average_temperature và land_max_temperature).
- Uncertainty (độ không chắc chắn) có thể không tương quan mạnh với các giá trị nhiệt độ nếu chất lượng đo lường không phụ thuộc vào mức độ nhiệt độ.
- **Ý nghĩa cho Feature Engineering:** Nếu tương quan rất cao (> 0.95), có thể xảy ra multicollinearity, cần cân nhắc loại bỏ một trong các biến.

### 3.2 Target Analysis (Phân tích biến mục tiêu)

#### 3.2.1 Xác định biến mục tiêu

Đối với dữ liệu nhiệt độ, **biến mục tiêu (target)** có thể là:
- `average_temperature` (nhiệt độ trung bình) nếu dự báo cho mỗi thành phố
- `year` nếu dự báo trên dữ liệu bình quân toàn cầu

In [ ]:
if df is not None:
    # Xác định cột mục tiêu (thường là average_temperature)
    target_col = None
    if 'average_temperature' in df.columns:
        target_col = 'average_temperature'
    elif 'avg_temperature' in df.columns:
        target_col = 'avg_temperature'
    
    if target_col:
        print(f"✓ Biến mục tiêu (Target): {target_col}")
        
        # Loại bỏ NaN
        target_data = df[target_col].dropna()
        
        # Thống kê mô tả
        print(f"\n📊 Thống kê của {target_col}:")
        print(f"   Số lượng: {len(target_data):,}")
        print(f"   Trung bình (Mean): {target_data.mean():.2f}°C")
        print(f"   Trung vị (Median): {target_data.median():.2f}°C")
        print(f"   Độ lệch chuẩn (Std): {target_data.std():.2f}")
        print(f"   Min: {target_data.min():.2f}°C")
        print(f"   Max: {target_data.max():.2f}°C")
        print(f"   Khoảng (Range): {target_data.max() - target_data.min():.2f}°C")
        
        # Xác định số bins phù hợp
        n_data = len(target_data)
        n_bins = max(20, min(100, n_data // 100 + 30))
        
        # Histogram
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram
        axes[0].hist(target_data, bins=n_bins, color='steelblue', edgecolor='black', alpha=0.7)
        axes[0].axvline(target_data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {target_data.mean():.2f}°C')
        axes[0].axvline(target_data.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {target_data.median():.2f}°C')
        axes[0].set_xlabel(target_col, fontsize=11)
        axes[0].set_ylabel('Tần số (Frequency)', fontsize=11)
        axes[0].set_title(f'Phân bố {target_col} (Histogram)', fontsize=12, fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Boxplot
        axes[1].boxplot(target_data, vert=True)
        axes[1].set_ylabel(target_col, fontsize=11)
        axes[1].set_title(f'Phân bố {target_col} (Boxplot)', fontsize=12, fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '02_target_distribution.png', dpi=300, bbox_inches='tight')
            print("\n✓ Đã lưu: reports/images/02_target_distribution.png")
        
        plt.show()
    else:
        print("⚠ Không tìm thấy cột mục tiêu (average_temperature hoặc avg_temperature)")

#### 3.2.2 Nhận xét về phân bố biến mục tiêu

**Quan sát:**
- **Hình dạng phân bố:** Bình thường, lệch trái/phải, hoặc nhiều peak?
- **Outliers:** Có nhiệt độ bất thường (quá cao/quá thấp) không?
- **Ý nghĩa:** Phân bố cân bằng là tốt cho mô hình hồi quy. Nếu lệch, có thể cần transform (log, sqrt).
- **Ứng dụng cho ML:** Nếu phân bố bình thường, các mô hình tuyến tính sẽ hiệu quả hơn.

### 3.3 Univariate Analysis (Phân tích từng biến riêng lẻ)

In [ ]:
if df is not None:
    # Phân tích các cột danh mục (categorical)
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    
    if categorical_cols:
        print(f"📋 Phân tích các cột danh mục: {categorical_cols}")
        
        # Tính kích thước động
        n_categories = len(categorical_cols)
        max_items = max(df[col].nunique() for col in categorical_cols)
        row_height = min(max(3, max_items * 0.15), 8)  # 3-8 inches per row
        fig_height = n_categories * row_height
        
        fig, axes = plt.subplots(n_categories, 1, figsize=(12, fig_height))
        if n_categories == 1:
            axes = [axes]
        
        for idx, col in enumerate(categorical_cols):
            value_counts = df[col].value_counts().head(20)  # Top 20
            
            print(f"\n🔹 {col}:")
            print(f"   Số giá trị duy nhất: {df[col].nunique()}")
            print(f"   Missing: {df[col].isna().sum()}")
            print(f"   Top 5:")
            for val, count in value_counts.head(5).items():
                print(f"      {val}: {count}")
            
            # Biểu đồ thanh
            axes[idx].barh(range(len(value_counts)), value_counts.values, color='steelblue')
            axes[idx].set_yticks(range(len(value_counts)))
            axes[idx].set_yticklabels(value_counts.index, fontsize=9)
            axes[idx].set_xlabel('Count', fontsize=10)
            axes[idx].set_title(f'Phân bố {col}', fontsize=11, fontweight='bold')
            axes[idx].grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '03_categorical_analysis.png', dpi=300, bbox_inches='tight')
            print("\n✓ Đã lưu: reports/images/03_categorical_analysis.png")
        
        plt.show()

### 3.4 Correlation with Target Variable (Tương quan với biến mục tiêu)

In [ ]:
if df is not None and target_col:
    # Tính tương quan của các biến số với target
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        correlations = df[numeric_cols].corr()[target_col].sort_values(ascending=False)
        
        print(f"\n📊 Tương quan của các biến với {target_col}:")
        print(correlations)
        
        # Tính kích thước phù hợp
        n_vars = len(correlations)
        fig_height = max(5, n_vars * 0.3)
        
        # Biểu đồ thanh
        fig, ax = plt.subplots(figsize=(10, fig_height))
        colors = ['green' if x > 0 else 'red' for x in correlations.values]
        ax.barh(range(len(correlations)), correlations.values, color=colors, alpha=0.7)
        ax.set_yticks(range(len(correlations)))
        ax.set_yticklabels(correlations.index, fontsize=10)
        ax.set_xlabel('Correlation Coefficient', fontsize=11)
        ax.set_title(f'Tương quan với {target_col}', fontsize=12, fontweight='bold')
        ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '04_correlation_with_target.png', dpi=300, bbox_inches='tight')
            print("✓ Đã lưu: reports/images/04_correlation_with_target.png")
        
        plt.show()
        
        # Nhận xét
        print("\n💡 Nhận xét:")
        strong_corr = correlations[correlations.abs() > 0.7]
        if len(strong_corr) > 0:
            print(f"   Biến có tương quan mạnh (|r| > 0.7):")
            for var, corr in strong_corr.items():
                print(f"   - {var}: {corr:.3f}")
        else:
            print(f"   Không có biến nào có tương quan mạnh (|r| > 0.7) với {target_col}")

### 3.5 Multivariate Analysis (Phân tích tương tác giữa các biến)

#### 3.5.1 Phân tích từng cặp biến chính

In [ ]:
if df is not None and target_col:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Loại bỏ target nếu nó trong danh sách
    feature_cols = [col for col in numeric_cols if col != target_col][:6]  # Lấy tối đa 6 biến
    
    if feature_cols:
        # Tính kích thước động
        n_features = len(feature_cols)
        col_width = 5
        total_width = n_features * col_width
        
        fig, axes = plt.subplots(1, n_features, figsize=(total_width, 4.5))
        if n_features == 1:
            axes = [axes]
        
        for idx, col in enumerate(feature_cols):
            # Loại bỏ NaN
            valid_data = df[[col, target_col]].dropna()
            
            axes[idx].scatter(valid_data[col], valid_data[target_col], alpha=0.5, s=20)
            axes[idx].set_xlabel(col, fontsize=10)
            axes[idx].set_ylabel(target_col, fontsize=10)
            axes[idx].set_title(f'{col} vs {target_col}', fontsize=11, fontweight='bold')
            axes[idx].grid(True, alpha=0.3)
            
            # Thêm đường trend
            if len(valid_data) > 1:
                z = np.polyfit(valid_data[col], valid_data[target_col], 1)
                p = np.poly1d(z)
                x_line = np.linspace(valid_data[col].min(), valid_data[col].max(), 100)
                axes[idx].plot(x_line, p(x_line), "r-", alpha=0.8, linewidth=2)
        
        plt.tight_layout()
        
        # Lưu ảnh
        if REPORT_IMAGES.exists():
            plt.savefig(REPORT_IMAGES / '05_multivariate_scatter.png', dpi=300, bbox_inches='tight')
            print("✓ Đã lưu: reports/images/05_multivariate_scatter.png")
        
        plt.show()

### 3.6 Time Series Analysis (Phân tích theo thời gian)

#### 3.6.1 Xác định cột thời gian

In [ ]:
if df is not None:
    # Tìm cột thời gian
    date_cols = [col for col in df.columns if 'date' in col.lower() or col.lower() in ['year', 'month', 'day']]
    
    print(f"🕐 Các cột thời gian tìm thấy: {date_cols}")
    
    # Xác định cột thời gian chính
    date_col = None
    if 'observation_date' in df.columns:
        date_col = 'observation_date'
    elif 'year' in df.columns:
        date_col = 'year'
    elif date_cols:
        date_col = date_cols[0]
    
    if date_col:
        print(f"✓ Sử dụng cột thời gian: {date_col}")

🕐 Các cột thời gian tìm thấy: ['observation_date', 'year', 'month']
✓ Sử dụng cột thời gian: observation_date


#### 3.6.2 Xu hướng theo năm

In [ ]:
if df is not None and target_col and 'year' in df.columns:
    # Tính trung bình theo năm
    yearly_avg = df.groupby('year')[target_col].agg(['mean', 'std', 'count']).reset_index()
    
    print(f"\n📈 Xu hướng {target_col} theo năm:")
    print(yearly_avg.head(10))
    
    # Tính kích thước phù hợp
    n_years = len(yearly_avg)
    fig_width = max(12, n_years * 0.08 + 2)
    
    # Biểu đồ đường
    fig, ax = plt.subplots(figsize=(fig_width, 6))
    ax.plot(yearly_avg['year'], yearly_avg['mean'], marker='o', linewidth=2, markersize=4, label='Mean')
    ax.fill_between(yearly_avg['year'], 
                     yearly_avg['mean'] - yearly_avg['std'], 
                     yearly_avg['mean'] + yearly_avg['std'], 
                     alpha=0.2, label='±1 Std Dev')
    ax.set_xlabel('Năm (Year)', fontsize=11)
    ax.set_ylabel(f'{target_col} (°C)', fontsize=11)
    ax.set_title(f'Xu hướng {target_col} theo Năm', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Lưu ảnh
    if REPORT_IMAGES.exists():
        plt.savefig(REPORT_IMAGES / '06_yearly_trend.png', dpi=300, bbox_inches='tight')
        print("\n✓ Đã lưu: reports/images/06_yearly_trend.png")
    
    plt.show()
    
    # Tính xu hướng
    trend_slope = (yearly_avg['mean'].iloc[-1] - yearly_avg['mean'].iloc[0]) / len(yearly_avg)
    print(f"\n📊 Xu hướng: {trend_slope:.4f}°C/năm")
    if trend_slope > 0:
        print(f"   ⬆️  Nhiệt độ tăng theo thời gian (nóng lên)")
    else:
        print(f"   ⬇️  Nhiệt độ giảm theo thời gian (lạnh lên)")

#### 3.6.3 Mùa vụ (nếu có cột tháng)

In [ ]:
if df is not None and target_col and 'month' in df.columns:
    # Tính trung bình theo tháng
    monthly_avg = df.groupby('month')[target_col].agg(['mean', 'std', 'count']).reset_index()
    monthly_avg = monthly_avg.sort_values('month')
    
    print(f"\n🔄 Mùa vụ - Trung bình {target_col} theo Tháng:")
    print(monthly_avg)
    
    # Kích thước phù hợp cho 12 tháng
    fig, ax = plt.subplots(figsize=(13, 6))
    colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(monthly_avg)))
    ax.bar(monthly_avg['month'], monthly_avg['mean'], color=colors, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Tháng (Month)', fontsize=11)
    ax.set_ylabel(f'{target_col} (°C)', fontsize=11)
    ax.set_title(f'Mùa vụ - Trung bình {target_col} theo Tháng', fontsize=12, fontweight='bold')
    ax.set_xticks(range(1, 13))
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Lưu ảnh
    if REPORT_IMAGES.exists():
        plt.savefig(REPORT_IMAGES / '07_monthly_seasonality.png', dpi=300, bbox_inches='tight')
        print("\n✓ Đã lưu: reports/images/07_monthly_seasonality.png")
    
    plt.show()
    
    # Nhận xét
    warmest_month = monthly_avg.loc[monthly_avg['mean'].idxmax()]
    coldest_month = monthly_avg.loc[monthly_avg['mean'].idxmin()]
    print(f"\n🌡️ Nhận xét:")
    print(f"   - Tháng nóng nhất: Tháng {int(warmest_month['month'])} ({warmest_month['mean']:.2f}°C)")
    print(f"   - Tháng lạnh nhất: Tháng {int(coldest_month['month'])} ({coldest_month['mean']:.2f}°C)")
    print(f"   - Chênh lệch: {warmest_month['mean'] - coldest_month['mean']:.2f}°C")

---

## IV. Kết Luận EDA

### 4.1 Các Phát Hiện Chính

Sau khi phân tích dữ liệu nhiệt độ bề mặt Trái Đất, những phát hiện chính bao gồm:

**1. Phân bố dữ liệu:**
   - Dữ liệu nhiệt độ tuân theo phân bố [bình thường / lệch / ...]
   - Có/Không có outliers đáng chú ý

**2. Xu hướng theo thời gian:**
   - Xu hướng nóng lên/lạnh lên rõ ràng từ năm X đến năm Y
   - Tốc độ thay đổi: [X°C/năm]

**3. Mùa vụ:**
   - Có sự biến động rõ rệt giữa các tháng
   - Chênh lệch giữa tháng nóng nhất và tháng lạnh nhất: [X°C]

**4. Tương quan giữa các biến:**
   - Các biến nhiệt độ (avg, max, min) có tương quan cao (r > 0.9)
   - Uncertainty không có tương quan mạnh với các giá trị nhiệt độ

**5. Khác biệt địa lý (nếu có dữ liệu theo quốc gia/thành phố):**
   - Một số khu vực nóng lên nhanh hơn những nơi khác
   - Có sự phân tán lớn giữa các địa điểm

**6. Chất lượng dữ liệu:**
   - Tỷ lệ missing values: [X%]
   - Dữ liệu đã được làm sạch (từ Notebook 03)

### 4.2 Ý Nghĩa cho Dự Báo

- **Tính mùa vụ rõ rệt:** Cần tạo các đặc trưng theo tháng/quý (seasonal decomposition)
- **Xu hướng dài hạn:** Cần tạo polynomial features hoặc moving average
- **Tương quan cao:** Có thể sử dụng PCA để giảm dimensionality
- **Sự khác biệt địa lý:** Cần xem xét encoding theo quốc gia/khu vực

### 4.3 Các Kích Hoạt cho Feature Engineering

Những phát hiện này sẽ được sử dụng để:
1. **Tạo đặc trưng thời gian:** `month`, `quarter`, `season`, `day_of_year`, v.v.
2. **Tạo đặc trưng xu hướng:** `year_since_start`, polynomial features
3. **Tạo đặc trưng tương tác:** nếu cần interaction giữa tháng và năm
4. **Xử lý multicollinearity:** loại bỏ các biến dư thừa
5. **Xử lý địa lý:** encoding theo quốc gia/vùng nếu phù hợp

---

## V. Liên Kết Sang Notebook 05: Feature Engineering

### 5.1 Tóm Tắt

Notebook 04 đã hoàn thành EDA và phát hiện:
- ✅ Các đặc điểm cơ bản của dữ liệu
- ✅ Mối quan hệ giữa các biến
- ✅ Xu hướng theo thời gian
- ✅ Mùa vụ và tính chất thời gian

### 5.2 Chuẩn Bị cho Notebook 05

**Notebook 05 (Feature Engineering) sẽ:**
1. Sử dụng các phát hiện từ EDA để tạo đặc trưng mới
2. Loại bỏ đặc trưng dư thừa (multicollinearity)
3. Tạo polynomial features, lag features, rolling averages
4. Chuẩn bị dữ liệu cho Notebook 06 (Machine Learning)

**Các đặc trưng dự kiến:**
- Time-based: `month`, `quarter`, `season`, `is_summer`, `is_winter`
- Trend: `year_normalized`, polynomial terms, moving averages
- Lag: previous month/year temperature
- Statistical: rolling mean, rolling std

### 5.3 Quy Trình Tiếp Theo

```
04_eda_visualization (Hiện tại) ✓
         ↓
05_feature_engineering (Tiếp theo)
         ↓
06_machine_learning (Train mô hình)
         ↓
07_prediction_demo (Thực hiện dự báo)
```

---

## VI. Ghi Chú và Hướng Dẫn Chạy Notebook

### Cách Chạy Notebook

1. **Đảm bảo Notebook 03 đã chạy:** để tạo file `cleaned_city_temperature.csv` hoặc table PostgreSQL
2. **Chạy từ trên xuống (Run All):** Ctrl+Shift+P → "Run All Cells"
3. **Kiểm tra output:** Xem messages để xác nhận dữ liệu đã được tải
4. **Kiểm tra ảnh:** Các biểu đồ sẽ được lưu trong `reports/images/`

### Xử Lý Lỗi Thường Gặp

| Lỗi | Nguyên Nhân | Giải Pháp |
|---|---|---|
| `ModuleNotFoundError: No module named 'psycopg2'` | Thiếu PostgreSQL driver | `pip install psycopg2-binary python-dotenv` |
| `FileNotFoundError: cleaned_city_temperature.csv` | Notebook 03 chưa chạy | Chạy Notebook 03 trước |
| `ConnectionError` từ PostgreSQL | Database không chạy | Kiểm tra PostgreSQL, hoặc notebook sẽ fallback sang CSV |
| Biểu đồ không hiển thị | Jupyter kernel hang | Restart kernel |

### Tùy Chỉnh

- **Thay đổi biến mục tiêu:** Sửa `target_col = 'average_temperature'` thành cột khác
- **Thay đổi số lượng bins:** Sửa `bins=50` trong histogram
- **Lưu thêm ảnh:** Thêm `plt.savefig(REPORT_IMAGES / 'tên_file.png')` trước `plt.show()`